# AI vs. Human Text Classifier
Statistical/feature-based approach using GPT-2 perplexity, burstiness, readability scores, POS ratios, and lexical features.
Data: HC3 + GPT-wiki-intro (HuggingFace). Model: best of LR / RF / XGBoost.

## 0. Setup

In [ ]:
# Uncomment on first run
# !pip install -r ../requirements.txt
# !python -m spacy download en_core_web_sm

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset
import nltk

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add src/ to path so we can import features.py
sys.path.insert(0, os.path.abspath("../src"))
from features import (
    lexical_features, readability_features,
    pos_features_from_doc, compute_pos_features_batch,
    compute_perplexity, compute_perplexity_batch,
    compute_all_features, _ensure_nltk
)

_ensure_nltk()
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_DIR      = "../data/raw"
PROCESSED_DIR = "../data/processed"
MODEL_DIR     = "../models"
FIGURE_DIR    = "../reports/figures"
for d in [DATA_DIR, PROCESSED_DIR, MODEL_DIR, FIGURE_DIR]:
    os.makedirs(d, exist_ok=True)
print("Setup complete.")

## 1. Data Loading

In [ ]:
# --- HC3 ---
print("Loading HC3...")
hc3_raw = load_dataset("Hello-SimpleAI/HC3", "all", cache_dir=DATA_DIR, trust_remote_code=True)
hc3_df  = hc3_raw["train"].to_pandas()

hc3_human = hc3_df["human_answers"].explode().dropna()
hc3_human = hc3_human[hc3_human.str.strip().str.len() > 50]
hc3_human = pd.DataFrame({"text": hc3_human.values, "label": 0})

hc3_ai = hc3_df["chatgpt_answers"].explode().dropna()
hc3_ai = hc3_ai[hc3_ai.str.strip().str.len() > 50]
hc3_ai = pd.DataFrame({"text": hc3_ai.values, "label": 1})

print(f"HC3  — human: {len(hc3_human):,}  ai: {len(hc3_ai):,}")

In [ ]:
# --- GPT-wiki-intro ---
print("Loading GPT-wiki-intro...")
wiki_raw = load_dataset("aadityaubhat/GPT-wiki-intro", cache_dir=DATA_DIR)
wiki_df  = wiki_raw["train"].to_pandas()

wiki_human = pd.DataFrame({"text": wiki_df["wiki_intro"].values, "label": 0})
wiki_ai    = pd.DataFrame({"text": wiki_df["generated_intro"].values, "label": 1})

wiki_human = wiki_human[wiki_human["text"].str.strip().str.len() > 50]
wiki_ai    = wiki_ai[wiki_ai["text"].str.strip().str.len() > 50]

# Cap wiki at 5k/class to avoid encyclopedic-style bias
WIKI_CAP = 5_000
wiki_human = wiki_human.sample(min(WIKI_CAP, len(wiki_human)), random_state=RANDOM_STATE)
wiki_ai    = wiki_ai.sample(min(WIKI_CAP, len(wiki_ai)),    random_state=RANDOM_STATE)
print(f"Wiki — human: {len(wiki_human):,}  ai: {len(wiki_ai):,}")

In [ ]:
from langdetect import detect, LangDetectException

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

# Pool, deduplicate, balance, shuffle
all_human = pd.concat([hc3_human, wiki_human], ignore_index=True).drop_duplicates("text")
all_ai    = pd.concat([hc3_ai,    wiki_ai],    ignore_index=True).drop_duplicates("text")

N_SAMPLES = 20_000
all_human = all_human.sample(min(N_SAMPLES, len(all_human)), random_state=RANDOM_STATE)
all_ai    = all_ai.sample(min(N_SAMPLES, len(all_ai)),    random_state=RANDOM_STATE)

df = pd.concat([all_human, all_ai], ignore_index=True)
df["text"] = df["text"].astype(str).str.strip()

# Filter: ≥3 sentences, English only
print("Filtering short / non-English texts...")
df["sent_count_pre"] = df["text"].apply(lambda t: len(nltk.sent_tokenize(t)))
df = df[df["sent_count_pre"] >= 3].reset_index(drop=True)

print("Detecting language (sample check — may take a moment)...")
en_mask = df["text"].apply(is_english)
df = df[en_mask].reset_index(drop=True)
df = df.drop(columns=["sent_count_pre"])
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df["source"] = df["label"].map({0: "human", 1: "ai"})

print(f"Final dataset: {len(df):,} samples")
print(df["source"].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
df["source"].value_counts().plot.bar(ax=ax, color=["steelblue", "tomato"])
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/class_dist.png", dpi=150)
plt.show()

In [ ]:
df["char_len"]   = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
df["sent_count"] = df["text"].apply(lambda t: len(nltk.sent_tokenize(t)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["char_len", "word_count", "sent_count"]):
    for src, grp in df.groupby("source"):
        cap = grp[col].quantile(0.99)
        grp[col].clip(upper=cap).hist(ax=ax, bins=60, alpha=0.6, label=src, density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/length_dist.png", dpi=150)
plt.show()

In [ ]:
pd.set_option("display.max_colwidth", 300)
print("=== Human samples ===")
display(df[df["label"]==0].sample(3, random_state=0)[["text","word_count"]])
print("=== AI samples ===")
display(df[df["label"]==1].sample(3, random_state=0)[["text","word_count"]])

## 3. Feature Engineering
> **Note:** This section takes ~20–40 min on CPU due to GPT-2 perplexity. A checkpoint is saved to `data/processed/feature_matrix.parquet` after completion. Re-running just loads from disk.

In [ ]:
CHECKPOINT = f"{PROCESSED_DIR}/feature_matrix.parquet"

if os.path.exists(CHECKPOINT):
    print(f"Loading checkpoint: {CHECKPOINT}")
    feature_df = pd.read_parquet(CHECKPOINT)
    print(f"Loaded: {feature_df.shape}")
else:
    print("Computing features from scratch...")

    print("  Lexical features...")
    lexical_list = [lexical_features(t) for t in tqdm(df["text"], desc="Lexical")]
    lexical_df = pd.DataFrame(lexical_list)

    print("  Readability features...")
    read_list = [readability_features(t) for t in tqdm(df["text"], desc="Readability")]
    read_df = pd.DataFrame(read_list)

    print("  POS features (spaCy batched)...")
    pos_list = compute_pos_features_batch(df["text"].tolist())
    pos_df = pd.DataFrame(pos_list)

    print("  GPT-2 perplexity (slow step)...")
    ppls = compute_perplexity_batch(df["text"].tolist(), batch_size=16)

    feature_df = pd.concat(
        [lexical_df.reset_index(drop=True),
         read_df.reset_index(drop=True),
         pos_df.reset_index(drop=True)],
        axis=1
    )
    feature_df["gpt2_perplexity"] = ppls
    feature_df["label"] = df["label"].values

    # Fill NaNs with column median
    for col in feature_df.columns:
        if feature_df[col].isnull().any():
            feature_df[col] = feature_df[col].fillna(feature_df[col].median())

    feature_df.to_parquet(CHECKPOINT, index=False)
    print(f"Saved checkpoint: {feature_df.shape}")

In [ ]:
feature_cols = [c for c in feature_df.columns if c != "label"]
print(f"Total features: {len(feature_cols)}")
print(f"NaN counts:\n{feature_df.isnull().sum()[feature_df.isnull().sum() > 0]}")
feature_df.head(3)

## 4. Feature Analysis

In [ ]:
corr = feature_df[feature_cols].corrwith(feature_df["label"]).abs().sort_values(ascending=False)
print("Top 20 features by |correlation with label|:")
display(corr.head(20).to_frame("abs_corr").style.bar(color="steelblue"))

In [ ]:
top9 = corr.head(9).index.tolist()
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, feat in zip(axes.flat, top9):
    for label_val, src in [(0, "human"), (1, "ai")]:
        vals = feature_df.loc[feature_df["label"]==label_val, feat]
        vals = vals.clip(lower=vals.quantile(0.01), upper=vals.quantile(0.99))
        vals.hist(ax=ax, bins=50, alpha=0.6, label=src, density=True,
                  color="steelblue" if src=="human" else "tomato")
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/feature_distributions.png", dpi=150)
plt.show()

## 5. Train / Val / Test Split (60 / 20 / 20)

In [ ]:
X = feature_df[feature_cols].values
y = feature_df["label"].values

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=RANDOM_STATE)

print(f"Train : {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"Train class balance: {np.bincount(y_train)}")

## 6. Model Training

In [ ]:
pipelines = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, C=1.0,
                                    class_weight="balanced",
                                    random_state=RANDOM_STATE))
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                        class_weight="balanced",
                                        random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(n_estimators=300, max_depth=6,
                               learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8,
                               eval_metric="logloss",
                               random_state=RANDOM_STATE, n_jobs=-1))
    ]),
}

trained = {}
for name, pipe in pipelines.items():
    print(f"Training {name}...")
    if name == "XGBoost":
        # Scale val set manually just to pass as eval_set
        scaler_temp = pipe.named_steps["scaler"]
        pipe.fit(X_train, y_train,
                 clf__eval_set=[(scaler_temp.fit_transform(X_val), y_val)],
                 clf__verbose=False)
    else:
        pipe.fit(X_train, y_train)
    trained[name] = pipe
    val_acc = accuracy_score(y_val, pipe.predict(X_val))
    val_f1  = f1_score(y_val, pipe.predict(X_val), average="macro")
    print(f"  val accuracy={val_acc:.4f}  val F1-macro={val_f1:.4f}")

## 7. Cross-Validation on Best Model

In [ ]:
best_name = max(trained, key=lambda n: f1_score(y_val, trained[n].predict(X_val), average="macro"))
print(f"Best model (by val F1-macro): {best_name}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(trained[best_name], X_trainval, y_trainval,
                             cv=cv, scoring="f1_macro", n_jobs=-1)
print(f"5-fold CV F1-macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 8. Evaluation on Test Set

In [ ]:
results = []
for name, pipe in trained.items():
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "model":    name,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "f1_ai":    f1_score(y_test, y_pred, pos_label=1, average="binary"),
        "f1_human": f1_score(y_test, y_pred, pos_label=0, average="binary"),
        "roc_auc":  roc_auc_score(y_test, y_proba),
    })

results_df = (pd.DataFrame(results)
               .set_index("model")
               .sort_values("f1_macro", ascending=False))
display(results_df.style.highlight_max(axis=0, color="lightgreen").format("{:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, len(trained), figsize=(5 * len(trained), 4))
if len(trained) == 1:
    axes = [axes]
for ax, (name, pipe) in zip(axes, trained.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="Blues",
                xticklabels=["human","ai"], yticklabels=["human","ai"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, pipe in trained.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
ax.plot([0,1],[0,1],"k--", label="Random")
ax.set_title("ROC Curves — Test Set")
ax.legend()
plt.savefig(f"{FIGURE_DIR}/roc_curves.png", dpi=150)
plt.show()

In [ ]:
print(f"=== Classification Report: {best_name} ===")
print(classification_report(y_test, trained[best_name].predict(X_test),
                             target_names=["human", "ai"]))

## 9. Feature Importance

In [ ]:
if "RandomForest" in trained:
    rf_clf = trained["RandomForest"].named_steps["clf"]
    rf_imp = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    rf_imp.head(20).plot.bar(ax=ax, color="steelblue")
    ax.set_title("Top 20 Feature Importances (Random Forest)")
    ax.set_ylabel("Mean Decrease in Impurity")
    plt.tight_layout()
    plt.savefig(f"{FIGURE_DIR}/rf_feature_importance.png", dpi=150)
    plt.show()

In [ ]:
if "XGBoost" in trained:
    xgb_clf = trained["XGBoost"].named_steps["clf"]
    xgb_imp = pd.Series(xgb_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    xgb_imp.head(20).plot.bar(ax=ax, color="tomato")
    ax.set_title("Top 20 Feature Importances (XGBoost)")
    plt.tight_layout()
    plt.savefig(f"{FIGURE_DIR}/xgb_feature_importance.png", dpi=150)
    plt.show()

In [ ]:
if "LogisticRegression" in trained:
    lr_clf  = trained["LogisticRegression"].named_steps["clf"]
    lr_coef = pd.Series(lr_clf.coef_[0], index=feature_cols).sort_values()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    lr_coef.tail(15).plot.barh(ax=axes[0], color="tomato",   title="Top 15 → AI")
    lr_coef.head(15).plot.barh(ax=axes[1], color="steelblue", title="Top 15 → Human")
    plt.tight_layout()
    plt.savefig(f"{FIGURE_DIR}/lr_coefficients.png", dpi=150)
    plt.show()

## 10. Model Serialization

In [ ]:
best_pipe = trained[best_name]

model_path    = f"{MODEL_DIR}/best_model.joblib"
metadata_path = f"{MODEL_DIR}/model_metadata.joblib"

joblib.dump(best_pipe, model_path)

metadata = {
    "model_name":    best_name,
    "feature_cols":  feature_cols,
    "label_map":     {0: "human", 1: "ai"},
    "val_f1_macro":  float(f1_score(y_val,  best_pipe.predict(X_val),  average="macro")),
    "test_f1_macro": float(f1_score(y_test, best_pipe.predict(X_test), average="macro")),
}
joblib.dump(metadata, metadata_path)

print(f"Saved: {model_path}")
print(metadata)

In [ ]:
# Round-trip verification
loaded      = joblib.load(model_path)
loaded_meta = joblib.load(metadata_path)
np.testing.assert_array_equal(loaded.predict(X_test[:20]), best_pipe.predict(X_test[:20]))
print("Round-trip verification passed.")
print(f"Model expects {len(loaded_meta['feature_cols'])} features.")

In [ ]:
# Inference example — template for the future UI
def predict_text(text: str, model=loaded, meta=loaded_meta) -> dict:
    """Predict whether a text is AI-generated or human-written."""
    all_feats = compute_all_features(text)
    X_new = np.array([[all_feats[col] for col in meta["feature_cols"]]])
    pred  = model.predict(X_new)[0]
    proba = model.predict_proba(X_new)[0][pred]
    return {
        "label":       meta["label_map"][pred],
        "probability": float(proba),
        "confidence":  "high" if proba > 0.80 else "medium" if proba > 0.60 else "low",
    }

# Quick smoke test on two samples
sample_ai    = df[df["label"]==1].iloc[0]["text"]
sample_human = df[df["label"]==0].iloc[0]["text"]

print("AI sample:",    predict_text(sample_ai))
print("Human sample:", predict_text(sample_human))